In [ ]:
!python --version

## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

sample  
 ├── primary_site       (organ)  
 ├── disease_type       (broad class)  
 ├── subtype_global     (cross-cancer)  
 ├── subtype_tissue     (within-cancer)  
 └── histology          (biological family)  

### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

### 🧠 Big picture (this is powerful)

- You’ve essentially built the backbone of:
- cBioPortal-like explorer
- cohort stratification engine
- patient-specific pipeline (your perturbation_agent 🔥)

### 💡 Flow

> project → project_id  
> primary_sites → pid and disease_type  
> subtypes → subtype_id  
> stages →  stage_id  
> case_id → case_ids or UUIDs  
> samples → sample_id [tumor, normal]  
> aliquots → aliquot_id  
> files → file_id  


In [ ]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


### 👉 a TCGA cohort builder + data retrieval engine

Which means you can easily extend to:


> pan-cancer analyses  
> patient-specific pipelines (your perturb_agent 🔥)  
> automated workflows (Nextflow-ready)  

#### 🚀 Next steps

> Production pipeline  
  - CLI tool
  - cached queries
  - reproducible outputs

> Expression matrix builder  
  - auto-download
  - merge TSVs
  - DESeq2-ready matrix

> Patient-level analysis
  - per-case DEG
  - pathway perturbation scoring

### Defaults

In [ ]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)[:10]


### Subtypes explanation

👉 GDC does NOT give you clean biological subtypes
👉 You must derive them yourself

This is actually a feature, not a bug — because:

> you control granularity  
> you can harmonize across cancers  
> you avoid noisy labels  


💡 If you want to level this up

I can help you build:

🔬 A cross-TCGA subtype harmonizer
maps all projects → unified subtype ontology
handles synonyms (adenocarcinoma, NOS, etc.)
outputs clean ML-ready labels

👉 That would massively improve your perturbation analysis.

### Get all programs

In [ ]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


In [ ]:
np.array(prog_list)

### Primary sites given a program

In [ ]:
gdc.url_gdc_project

In [ ]:
force=False
verbose=False

pid = 'TCGA'

df_all_cases, df_all_samples, df_all_mutations = \
    gdc.loop_program_psi_samples(program=pid, force=force, verbose=verbose)

print("\n----------- end ------------\n")
print(len(df_all_samples))


In [ ]:
print(len(df_all_cases))
df_all_cases.tail(3)

In [ ]:
print(len(df_all_samples))
df_all_samples.tail(3)

In [ ]:
barcodes = np.unique(df_all_samples.barcode_id)
len(barcodes)

In [ ]:
sample_types = np.unique(df_all_samples.sample_type)
sample_types

In [ ]:
df_normal = df_all_samples[df_all_samples.sample_type.str.contains('Normal', case=False, na=False)]
len(df_normal)

In [ ]:
symbols = np.unique(df_all_mutations.symbol)
len(symbols)

In [ ]:
primary_sites = list(np.unique(df_all_cases.primary_site_norm))
primary_sites.sort()
print(len(primary_sites))

print("\n".join(primary_sites))

In [ ]:
root_summary = os.path.join(gdc.root_data, 'summary')
create_dir(root_summary)

In [ ]:
pid = 'TCGA'

In [ ]:
fname = f'{pid}_summ_cases.tsv'
pdwritecsv(df_all_cases, fname, root_summary)

fname = f'{pid}_summ_samples.tsv'
pdwritecsv(df_all_samples, fname, root_summary)

fname = f'{pid}_summ_mutations.tsv'
pdwritecsv(df_all_mutations, fname, root_summary)

In [ ]:
stri = f"Interfacing GDC {pid} data, one gathered:"
print(stri)
stri = f"\t- {len(primary_sites)} primary sites."
print(stri)
stri = f"\t- {len(df_all_cases)} cases."
print(stri)
stri = f"\t- {len(df_all_samples)} samples."
print(stri)
stri = f"\t- {len(df_all_mutations)} annotated mutations."
print(stri)
stri = f"\t- {len(symbols)} different genes."
print(stri)


### Development & Tests

In [ ]:
def loop_program_psi_samples(program:str='TCGA', ipsi:Any=None, force:bool=False,
        verbose:bool=True) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    
    df_psi = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

    df_cases, df_subt, df_prof = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    if isinstance(ipsi, int):
        lista = [ipsi]
    else:
        lista = np.arange(len(df_psi))

    df_list_cases, df_list_samples, df_list_mutations = [], [], []

    for ipsi in lista:
        row = df_psi.iloc[ipsi]
        pid = row.pid
        primary_site = row.primary_site

        print(f'{ipsi}) {primary_site}', end=' - ')

        df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)

        if df_cases.empty:
            print("No cases found for PID:", pid)
            continue

        if isinstance(df_cases, pd.DataFrame):
            df_list_cases.append(df_cases)
        else:
            print("Unexpected type for df_cases:", type(df_cases))
            raise Exception("Stope: unexpected type for df_cases")


        for isubt, row in df_subt.iterrows():
            subtype_global = row.subtype_global
            tumor_class = row.tumor_class
            subtype_tissue = row.subtype_tissue

            s_case = f"{pid}_{primary_site}_subtype_{subtype_global}_tumor_{tumor_class}_subtype_tissue_{subtype_tissue}"

            if len(s_case) > 180:
                s_case = f"{pid}_{primary_site[:40]}_subtype_{subtype_global[:40]}_tumor_{tumor_class[:40]}_tissue_{subtype_tissue[:40]}"

            s_case = title_replace(s_case)

            print(f'{isubt}) {s_case}')

            df_samples = gdc.get_samples_for_pid_subtypes(pid=pid, subtype_global=subtype_global,
                                                          tumor_class=tumor_class, subtype_tissue=subtype_tissue, s_case=s_case,
                                                          batch_size=200, force=force, verbose=verbose)
            
            if df_samples.empty:
                print("No samples found for PID:", pid)
                continue

            df_list_samples.append(df_samples)

            df2 = df_samples[~df_samples.sample_type.str.contains('Blood', case=False, na=False)]

            if df2.empty:
                print("No samples having non-blood types for PID:", pid)
                continue

            barcodes = list(np.unique(df2.barcode_id))

            print("Getting mutations", end=' ')
            dff, _ = gdc.get_df_mut_transform_mutation_table(study_id=pid, s_case=s_case, sample_ids=barcodes, force=force, verbose=verbose)

            if dff.empty:
                print("Could not find mutations for :", s_case)
                continue

            df_list_mutations.append(dff)

    if len(df_list_cases) > 0:
        df_all_cases = pd.concat(df_list_cases, ignore_index=True)
        df_all_cases = df_all_cases.drop_duplicates()
        df_all_cases = df_all_cases.reset_index(drop=True)
    else:
        df_all_cases = pd.DataFrame()

    if len(df_list_samples) > 0:
        df_all_samples = pd.concat(df_list_samples, ignore_index=True)
        df_all_samples = df_all_samples.drop_duplicates()
        df_all_samples = df_all_samples.reset_index(drop=True)
    else:
        df_all_samples = pd.DataFrame()

    if len(df_list_mutations) > 0:
        df_all_mutations = pd.concat(df_list_mutations, ignore_index=True)
        df_all_mutations = df_all_mutations.drop_duplicates()
        df_all_mutations = df_all_mutations.reset_index(drop=True)
    else:
        df_all_mutations = pd.DataFrame()        

    return df_psi, df_all_cases, df_all_samples, df_all_mutations

